# Unified Inference & Prediction Pipeline
# Spatiotemporal LSTM / Transformer / ST-Mamba

> Load trained models, run predictions, query specific coordinates, and export results.
> Supports all model architectures trained in this repository.

In [ ]:
import os, math, warnings, json, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from tqdm import tqdm
from scipy.spatial import cKDTree
from scipy.interpolate import griddata

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.parallel import DataParallel

from sklearn.preprocessing import StandardScaler

# ── INFERENCE CONFIG ─────────────────────────────────────────────────────────────────
INFERENCE_CONFIG = {
    # Data paths
    "pressure_csv": "/kaggle/input/master-2sec-updated/master_pressure_2sec.csv",
    "u_velocity_csv": "/kaggle/input/master-2sec-updated/master_u_velocity_2sec.csv",
    "v_velocity_csv": "/kaggle/input/master-2sec-updated/master_v_velocity_2sec.csv",

    # Model checkpoints (set to None if model not trained)
    "lstm_checkpoint": "outputs/best_model.pth",
    "transformer_checkpoint": None,  # set path if trained
    "st_mamba_checkpoint": None,     # set path if trained

    # Which model to use for primary inference
    "active_model": "SpatioTemporalLSTM",  # or "SpatioTemporalTransformer" or "STMamba"

    # Architecture params (must match training config)
    "spatial_embed_dim": 64,
    "hidden_size": 256,
    "num_layers": 3,
    "nhead": 8,
    "dropout": 0.2,
    "k_neighbors": 20,

    # Temporal params (must match training config)
    "input_sequence_length": 15,
    "prediction_horizon_steps": 5,
    "train_ratio": 0.60,
    "val_ratio": 0.20,

    # Inference settings
    "autoregressive_steps": 20,  # number of future steps to predict autoregressively
    "batch_size": 256,

    # Query coordinates (list of [x, y] pairs to extract predictions for)
    "query_coordinates": [],  # populated interactively or set manually

    # Output
    "output_dir": "inference_outputs",
    "export_csv": True,
    "create_visualizations": True,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
os.makedirs(INFERENCE_CONFIG["output_dir"], exist_ok=True)

## 1. Data Loading & Preprocessing

In [ ]:
def parse_urans_multifile(pressure_path, u_path, v_path, chunk_size=None):
    """
    Load three URANS CSV files (pressure, u-velocity, v-velocity).

    Each CSV has:
      - First 2 columns : x, y coordinates
      - Remaining columns: variable values at each timestep

    Returns
    -------
    coords          : np.ndarray, shape (n_cells, 2)
    data            : np.ndarray, shape (n_cells, n_timesteps, 3)  — order [P, U, V]
    n_timesteps     : int
    dataframes_dict : dict with keys 'pressure', 'u_velocity', 'v_velocity'
    """
    print("Loading CSV files ...")
    df_p = pd.read_csv(pressure_path)
    df_u = pd.read_csv(u_path)
    df_v = pd.read_csv(v_path)

    # Validate row counts
    assert len(df_p) == len(df_u) == len(df_v), (
        f"Row count mismatch: P={len(df_p)}, U={len(df_u)}, V={len(df_v)}"
    )
    n_cells = len(df_p)
    print(f"  n_cells = {n_cells:,}")

    # Extract coordinates (first 2 columns)
    coords_p = df_p.iloc[:, :2].values.astype(np.float64)
    coords_u = df_u.iloc[:, :2].values.astype(np.float64)
    coords_v = df_v.iloc[:, :2].values.astype(np.float64)

    assert np.allclose(coords_p, coords_u, atol=1e-8) and np.allclose(coords_p, coords_v, atol=1e-8), \
        "Coordinate mismatch across CSV files!"
    coords = coords_p  # shape (n_cells, 2)

    # Extract time-series data (columns 2+)
    ts_p = df_p.iloc[:, 2:].values.astype(np.float32)  # (n_cells, n_timesteps)
    ts_u = df_u.iloc[:, 2:].values.astype(np.float32)
    ts_v = df_v.iloc[:, 2:].values.astype(np.float32)

    assert ts_p.shape[1] == ts_u.shape[1] == ts_v.shape[1], (
        f"Timestep count mismatch: P={ts_p.shape[1]}, U={ts_u.shape[1]}, V={ts_v.shape[1]}"
    )
    n_timesteps = ts_p.shape[1]
    print(f"  n_timesteps = {n_timesteps}")

    # Apply chunk_size if requested
    if chunk_size is not None:
        n_cells = min(n_cells, chunk_size)
        coords = coords[:n_cells]
        ts_p = ts_p[:n_cells]
        ts_u = ts_u[:n_cells]
        ts_v = ts_v[:n_cells]
        print(f"  Using chunk_size={chunk_size}, effective n_cells={n_cells:,}")

    # Stack into (n_cells, n_timesteps, 3) — order [P, U, V]
    data = np.stack([ts_p, ts_u, ts_v], axis=-1)  # (n_cells, n_timesteps, 3)
    print(f"  data shape: {data.shape}")

    dataframes_dict = {
        "pressure": df_p,
        "u_velocity": df_u,
        "v_velocity": df_v,
    }
    return coords, data, n_timesteps, dataframes_dict

In [ ]:
def normalize_per_variable(data, train_end_t):
    """
    Normalize each variable (P, U, V) independently using StandardScaler.
    Scalers are fit only on the training time range to prevent data leakage.

    Parameters
    ----------
    data        : np.ndarray, shape (n_cells, n_timesteps, 3)
    train_end_t : int — exclusive end index of the training time range

    Returns
    -------
    data_normalized : np.ndarray, shape (n_cells, n_timesteps, 3)
    scalers         : list of 3 StandardScaler instances [scaler_p, scaler_u, scaler_v]
    """
    print("Normalizing per variable (fit on training split only) ...")
    n_cells, n_timesteps, n_vars = data.shape
    data_normalized = np.empty_like(data)
    scalers = []

    var_names = ["Pressure", "U-velocity", "V-velocity"]
    for var_idx in range(n_vars):
        scaler = StandardScaler()
        # Fit only on training timesteps: flatten (n_cells, train_end_t) → 1-D
        train_slice = data[:, :train_end_t, var_idx].reshape(-1, 1)
        scaler.fit(train_slice)
        # Transform all timesteps
        all_slice = data[:, :, var_idx].reshape(-1, 1)
        data_normalized[:, :, var_idx] = scaler.transform(all_slice).reshape(n_cells, n_timesteps)
        scalers.append(scaler)
        print(f"  {var_names[var_idx]}: mean={scaler.mean_[0]:.4f}, std={scaler.scale_[0]:.4f}")

    return data_normalized, scalers

In [ ]:
def build_knn_graph(coords, k):
    """
    For each cell, find its k nearest spatial neighbors (excluding self).

    Parameters
    ----------
    coords : np.ndarray, shape (n_cells, 2)
    k      : int — number of neighbors

    Returns
    -------
    neighbor_indices   : np.ndarray, shape (n_cells, k)
    neighbor_distances : np.ndarray, shape (n_cells, k)
    """
    print(f"Building kNN graph with k={k} ...")
    tree = cKDTree(coords)
    # Query k+1 because the first result is the point itself
    distances, indices = tree.query(coords, k=k + 1)
    neighbor_indices = indices[:, 1:]    # exclude self → (n_cells, k)
    neighbor_distances = distances[:, 1:]
    print(f"  neighbor_indices shape: {neighbor_indices.shape}")
    return neighbor_indices, neighbor_distances

## 2. Model Architecture Definitions

All model architectures must be defined identically to training to load checkpoints correctly.

In [ ]:
class DistanceWeightedSpatialEncoder(nn.Module):
    """
    Encodes the k-nearest-neighbor flow variables into a spatial embedding
    using distance-weighted aggregation instead of naive mean-pooling.

    A learned distance-scaling network (dist_scale) transforms raw spatial
    distances into per-neighbor weight logits. Weights are derived via
    inverse of these logits, so neighbors for which the network produces
    smaller logits receive larger weight. The network is free to learn any
    monotone mapping of distance, rather than using a fixed inverse-distance
    formula.
    """

    def __init__(self, input_features=3, embed_dim=64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_features, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )
        self.dist_scale = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Softplus(),
        )

    def forward(self, neighbor_seq, neighbor_dists):
        # neighbor_seq   : (B, seq_len, k, input_features)
        # neighbor_dists : (B, k)
        embedded = self.mlp(neighbor_seq)               # (B, seq_len, k, embed_dim)
        d = neighbor_dists.unsqueeze(-1)                # (B, k, 1)
        raw_w = self.dist_scale(d)                      # (B, k, 1)
        inv_w = 1.0 / (raw_w + 1e-6)                   # invert logits: smaller logit → larger weight
        weights = inv_w / (inv_w.sum(dim=1, keepdim=True) + 1e-8)  # (B, k, 1)
        weights = weights.unsqueeze(1)                  # (B, 1, k, 1)
        pooled = (embedded * weights).sum(dim=2)        # (B, seq_len, embed_dim)
        return pooled


class SpatioTemporalLSTM(nn.Module):
    """
    Spatiotemporal LSTM:
      1. DistanceWeightedSpatialEncoder aggregates neighbor features → spatial embedding
      2. Concatenate center-cell features with spatial embedding
      3. LSTM processes the combined sequence
      4. Decoder MLP maps last hidden state → (pred_steps, 3)
    """

    def __init__(self, input_features=3, spatial_embed_dim=64, hidden_size=256,
                 num_layers=3, pred_steps=15, dropout=0.2):
        super().__init__()
        self.pred_steps = pred_steps
        self.input_features = input_features
        self.spatial_encoder = DistanceWeightedSpatialEncoder(input_features, spatial_embed_dim)
        combined_dim = input_features + spatial_embed_dim
        self.lstm = nn.LSTM(
            combined_dim, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, pred_steps * input_features),
        )

    def forward(self, center_seq, neighbor_seq, neighbor_dists):
        # center_seq     : (B, seq_len, 3)
        # neighbor_seq   : (B, seq_len, k, 3)
        # neighbor_dists : (B, k)
        spatial_embed = self.spatial_encoder(neighbor_seq, neighbor_dists)          # (B, seq_len, spatial_embed_dim)
        combined = torch.cat([center_seq, spatial_embed], dim=-1)                   # (B, seq_len, 3+spatial_embed_dim)
        lstm_out, _ = self.lstm(combined)
        last_hidden = lstm_out[:, -1, :]                                            # (B, hidden_size)
        output = self.decoder(last_hidden)                                          # (B, pred_steps * 3)
        return output.view(-1, self.pred_steps, self.input_features)


class SpatioTemporalTransformer(nn.Module):
    """
    Spatiotemporal Transformer:
      1. DistanceWeightedSpatialEncoder aggregates neighbor features → spatial embedding
      2. Concatenate center-cell features with spatial embedding
      3. Linear projection to d_model + learned positional encoding
      4. Transformer encoder processes the sequence
      5. Decoder MLP maps last-position output → (pred_steps, 3)
    """

    def __init__(self, input_features=3, spatial_embed_dim=64, d_model=256,
                 nhead=8, num_layers=3, pred_steps=15, dropout=0.2):
        super().__init__()
        self.pred_steps = pred_steps
        self.input_features = input_features
        self.spatial_encoder = DistanceWeightedSpatialEncoder(input_features, spatial_embed_dim)
        combined_dim = input_features + spatial_embed_dim
        self.input_projection = nn.Linear(combined_dim, d_model)
        # Learned positional encoding (max 512 positions)
        self.pos_encoding = nn.Parameter(torch.randn(1, 512, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4, dropout=dropout, batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.decoder = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, pred_steps * input_features),
        )

    def forward(self, center_seq, neighbor_seq, neighbor_dists):
        # center_seq     : (B, seq_len, 3)
        # neighbor_seq   : (B, seq_len, k, 3)
        # neighbor_dists : (B, k)
        spatial_embed = self.spatial_encoder(neighbor_seq, neighbor_dists)          # (B, seq_len, spatial_embed_dim)
        combined = torch.cat([center_seq, spatial_embed], dim=-1)                   # (B, seq_len, 3+spatial_embed_dim)
        x = self.input_projection(combined)                                         # (B, seq_len, d_model)
        seq_len = x.shape[1]
        x = x + self.pos_encoding[:, :seq_len, :]                                   # add positional encoding
        x = self.transformer_encoder(x)                                             # (B, seq_len, d_model)
        last_output = x[:, -1, :]                                                   # (B, d_model)
        output = self.decoder(last_output)                                          # (B, pred_steps * 3)
        return output.view(-1, self.pred_steps, self.input_features)


def create_model(cfg):
    """
    Factory function that creates a model based on cfg['model_type'].

    Parameters
    ----------
    cfg : dict — CONFIG dictionary

    Returns
    -------
    model : nn.Module
    """
    model_type = cfg["model_type"]
    pred_steps = cfg["prediction_horizon_steps"]
    spatial_embed_dim = cfg["spatial_embed_dim"]
    hidden_size = cfg["hidden_size"]
    num_layers = cfg["num_layers"]
    dropout = cfg["dropout"]

    if model_type == "SpatioTemporalLSTM":
        model = SpatioTemporalLSTM(
            input_features=3,
            spatial_embed_dim=spatial_embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            pred_steps=pred_steps,
            dropout=dropout,
        )
    elif model_type == "SpatioTemporalTransformer":
        model = SpatioTemporalTransformer(
            input_features=3,
            spatial_embed_dim=spatial_embed_dim,
            d_model=hidden_size,
            nhead=cfg["nhead"],
            num_layers=num_layers,
            pred_steps=pred_steps,
            dropout=dropout,
        )
    else:
        raise ValueError(f"Unknown model_type: '{model_type}'. "
                         "Choose 'SpatioTemporalLSTM' or 'SpatioTemporalTransformer'.")

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Created {model_type} with {total_params:,} trainable parameters.")
    return model

## 3. Load Trained Model Checkpoint

In [ ]:
def load_trained_model(model_type, checkpoint_path, config, device):
    """
    Create model architecture and load trained weights.

    Parameters
    ----------
    model_type      : str — "SpatioTemporalLSTM" or "SpatioTemporalTransformer"
    checkpoint_path : str — path to .pth file
    config          : dict — must contain architecture hyperparameters
    device          : torch.device

    Returns
    -------
    model : nn.Module (in eval mode)
    """
    cfg = {
        "model_type": model_type,
        "prediction_horizon_steps": config["prediction_horizon_steps"],
        "spatial_embed_dim": config["spatial_embed_dim"],
        "hidden_size": config["hidden_size"],
        "num_layers": config["num_layers"],
        "nhead": config["nhead"],
        "dropout": config["dropout"],
    }
    model = create_model(cfg)
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    print(f"✓ Loaded {model_type} from {checkpoint_path}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    return model

## 4. Single-Step Prediction (All Cells)

Predict the next `prediction_horizon_steps` timesteps for every cell in the mesh
using the last `input_sequence_length` timesteps as input.

In [ ]:
@torch.no_grad()
def predict_all_cells(model, data_normalized, neighbor_indices, neighbor_distances, config, device):
    """
    For each cell, use the last seq_length timesteps as input,
    predict the next pred_steps timesteps.

    Returns
    -------
    predictions : np.ndarray, shape (n_cells, pred_steps, 3)  — normalized
    """
    model.eval()
    seq_len = config["input_sequence_length"]
    pred_steps = config["prediction_horizon_steps"]
    batch_size = config["batch_size"]
    n_cells = data_normalized.shape[0]
    n_timesteps = data_normalized.shape[1]

    data_tensor = torch.FloatTensor(data_normalized)
    nbr_tensor = torch.LongTensor(neighbor_indices)
    dist_tensor = torch.FloatTensor(neighbor_distances)

    all_preds = np.zeros((n_cells, pred_steps, 3), dtype=np.float32)

    for start in tqdm(range(0, n_cells, batch_size), desc="Predicting"):
        end = min(start + batch_size, n_cells)
        center_seq = data_tensor[start:end, n_timesteps - seq_len:, :]
        nbr_idx = nbr_tensor[start:end]
        neighbor_seq = data_tensor[nbr_idx, :, :]
        # Reorder: (batch, k, seq_len, 3) → (batch, seq_len, k, 3)
        neighbor_seq = neighbor_seq[:, :, n_timesteps - seq_len:, :].permute(0, 2, 1, 3)
        neighbor_dists = dist_tensor[start:end]

        center_seq = center_seq.to(device)
        neighbor_seq = neighbor_seq.to(device)
        neighbor_dists = neighbor_dists.to(device)

        pred = model(center_seq, neighbor_seq, neighbor_dists)
        all_preds[start:end] = pred.cpu().numpy()

    return all_preds

## 5. Autoregressive Multi-Step Rollout

For longer-horizon predictions beyond `prediction_horizon_steps`, 
feed each prediction back as input and roll forward step by step.

**How it works:**
1. Use last `seq_length` timesteps → predict next `pred_steps`
2. Append predictions to the history
3. Slide window forward by `pred_steps`
4. Repeat for `n_rollout_iterations`

In [ ]:
@torch.no_grad()
def autoregressive_rollout(model, data_normalized, neighbor_indices, neighbor_distances, config, device, n_rollout_steps=20):
    """
    Autoregressive multi-step prediction.

    At each iteration:
    1. Use the last seq_length timesteps from the current history
    2. Predict next pred_steps
    3. Append predictions to history
    4. Repeat

    Parameters
    ----------
    n_rollout_steps : int — total number of future timesteps to generate

    Returns
    -------
    all_rollout_preds : np.ndarray, shape (n_cells, n_rollout_steps, 3)
    """
    model.eval()
    seq_len = config["input_sequence_length"]
    pred_steps = config["prediction_horizon_steps"]
    batch_size = config["batch_size"]
    n_cells = data_normalized.shape[0]

    # Start with the full normalized data as history
    # We'll extend it with predictions
    history = torch.FloatTensor(data_normalized)  # (n_cells, n_timesteps, 3)
    nbr_tensor = torch.LongTensor(neighbor_indices)
    dist_tensor = torch.FloatTensor(neighbor_distances)

    all_rollout_preds = []
    steps_generated = 0

    iteration = 0
    while steps_generated < n_rollout_steps:
        iteration += 1
        current_n_timesteps = history.shape[1]

        # Predict next pred_steps for all cells
        batch_preds = []
        for start in range(0, n_cells, batch_size):
            end = min(start + batch_size, n_cells)
            center_seq = history[start:end, current_n_timesteps - seq_len:, :]
            nbr_idx = nbr_tensor[start:end]
            neighbor_seq = history[nbr_idx, :, :]
            # Reorder: (batch, k, seq_len, 3) → (batch, seq_len, k, 3)
            neighbor_seq = neighbor_seq[:, :, current_n_timesteps - seq_len:, :].permute(0, 2, 1, 3)
            neighbor_dists = dist_tensor[start:end]

            center_seq = center_seq.to(device)
            neighbor_seq = neighbor_seq.to(device)
            neighbor_dists = neighbor_dists.to(device)

            pred = model(center_seq, neighbor_seq, neighbor_dists)  # (batch, pred_steps, 3)
            batch_preds.append(pred.cpu())

        new_steps = torch.cat(batch_preds, dim=0)  # (n_cells, pred_steps, 3)

        # How many of these steps do we actually need?
        steps_to_take = min(pred_steps, n_rollout_steps - steps_generated)
        all_rollout_preds.append(new_steps[:, :steps_to_take, :].numpy())
        steps_generated += steps_to_take

        # Append to history for next iteration
        history = torch.cat([history, new_steps], dim=1)

        print(f"  Rollout iteration {iteration}: generated {steps_generated}/{n_rollout_steps} steps")

    return np.concatenate(all_rollout_preds, axis=1)  # (n_cells, n_rollout_steps, 3)

## 6. Query Specific Coordinates

Given any (x, y) coordinate (even one not in the mesh), find the nearest mesh cell
and return its predicted flow values. For points between cells, optionally interpolate
from the k nearest cells.

**Two modes:**
- **Nearest**: Return prediction from the single closest mesh cell
- **Interpolated**: Distance-weighted average from k nearest cells (smoother)

In [ ]:
class CoordinateQuery:
    """
    Query predictions at arbitrary (x, y) coordinates.

    Parameters
    ----------
    coords      : np.ndarray, shape (n_cells, 2) — mesh coordinates
    predictions : np.ndarray, shape (n_cells, n_steps, 3) — predicted P, U, V
    scalers     : list of 3 StandardScaler — for denormalization
    """

    def __init__(self, coords, predictions, scalers):
        self.coords = coords
        self.predictions = predictions  # normalized
        self.scalers = scalers
        self.tree = cKDTree(coords)
        self.n_cells, self.n_steps, _ = predictions.shape

    def query_nearest(self, x, y):
        """Find nearest mesh cell and return its denormalized predictions."""
        dist, idx = self.tree.query([x, y])
        pred_norm = self.predictions[idx]  # (n_steps, 3)
        pred_phys = self._denormalize(pred_norm)
        return {
            "query_point": (x, y),
            "nearest_cell_idx": idx,
            "nearest_cell_coords": self.coords[idx].tolist(),
            "distance": dist,
            "predictions": {
                "pressure": pred_phys[:, 0].tolist(),
                "u_velocity": pred_phys[:, 1].tolist(),
                "v_velocity": pred_phys[:, 2].tolist(),
            }
        }

    def query_interpolated(self, x, y, k=5):
        """
        Distance-weighted interpolation from k nearest cells.
        More accurate for points between mesh cells.
        """
        dists, indices = self.tree.query([x, y], k=k)

        # Inverse distance weighting (avoid division by zero)
        # DISTANCE_EPSILON avoids division by zero when query point coincides exactly with a mesh cell
        DISTANCE_EPSILON = 1e-10
        weights = 1.0 / (dists + DISTANCE_EPSILON)
        weights /= weights.sum()

        # Weighted average of predictions
        pred_norm = np.zeros((self.n_steps, 3))
        for w, idx in zip(weights, indices):
            pred_norm += w * self.predictions[idx]

        pred_phys = self._denormalize(pred_norm)
        return {
            "query_point": (x, y),
            "interpolation_cells": indices.tolist(),
            "interpolation_weights": weights.tolist(),
            "predictions": {
                "pressure": pred_phys[:, 0].tolist(),
                "u_velocity": pred_phys[:, 1].tolist(),
                "v_velocity": pred_phys[:, 2].tolist(),
            }
        }

    def query_batch(self, coordinates, mode="nearest", k=5):
        """
        Query multiple coordinates at once.

        Parameters
        ----------
        coordinates : list of [x, y] pairs
        mode        : "nearest" or "interpolated"
        """
        results = []
        for x, y in coordinates:
            if mode == "nearest":
                results.append(self.query_nearest(x, y))
            else:
                results.append(self.query_interpolated(x, y, k=k))
        return results

    def _denormalize(self, pred_norm):
        """Denormalize (n_steps, 3) prediction array."""
        pred_phys = np.zeros_like(pred_norm)
        for vi in range(3):
            pred_phys[:, vi] = self.scalers[vi].inverse_transform(
                pred_norm[:, vi].reshape(-1, 1)
            ).ravel()
        return pred_phys

## 7. Export Predictions to CSV

In [ ]:
def export_predictions_csv(coords, predictions, scalers, config, prefix="inference"):
    """
    Export denormalized predictions to 3 CSV files.

    Saves:
      {output_dir}/{prefix}_pressure.csv
      {output_dir}/{prefix}_u_velocity.csv
      {output_dir}/{prefix}_v_velocity.csv

    Format: x-coordinate, y-coordinate, pred_t1, pred_t2, ..., pred_tN
    """
    output_dir = config["output_dir"]
    os.makedirs(output_dir, exist_ok=True)

    n_cells, n_steps, _ = predictions.shape
    var_keys = ["pressure", "u_velocity", "v_velocity"]
    var_labels = ["P", "U", "V"]

    for vi, (key, label) in enumerate(zip(var_keys, var_labels)):
        pred_norm = predictions[:, :, vi]  # (n_cells, n_steps)
        pred_denorm = scalers[vi].inverse_transform(
            pred_norm.reshape(-1, 1)
        ).reshape(n_cells, n_steps)

        df = pd.DataFrame(coords, columns=["x-coordinate", "y-coordinate"])
        for t in range(n_steps):
            df[f"pred_{label}_t{t+1}"] = pred_denorm[:, t]

        out_path = os.path.join(output_dir, f"{prefix}_{key}.csv")
        try:
            df.to_csv(out_path, index=False)
            print(f"  Saved {out_path}  ({n_cells:,} cells × {n_steps} steps)")
        except OSError as exc:
            print(f"  ERROR: Could not write {out_path}: {exc}")

## 8. Visualization

### 8A. Time-series at queried coordinates
### 8B. Spatial field contour plots
### 8C. Error distribution analysis

In [ ]:
def plot_coordinate_prediction(query_result, config):
    """Plot P, U, V time series at a queried coordinate."""
    qx, qy = query_result["query_point"]
    preds = query_result["predictions"]
    n_steps = len(preds["pressure"])
    t = np.arange(1, n_steps + 1)

    fig, axes = plt.subplots(3, 1, figsize=(10, 8))
    fig.suptitle(f"Predictions at ({qx:.4f}, {qy:.4f})", fontsize=13)

    for ax, (key, label) in zip(axes, [("pressure", "Pressure (P)"),
                                        ("u_velocity", "U-velocity"),
                                        ("v_velocity", "V-velocity")]):
        ax.plot(t, preds[key], marker="o", markersize=3, linewidth=1.5)
        ax.set_ylabel(label)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("Predicted Timestep")
    plt.tight_layout()

    cell_idx = query_result.get("nearest_cell_idx", "interp")
    out_path = os.path.join(config["output_dir"], f"coord_pred_cell_{cell_idx}.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved {out_path}")


def plot_spatial_field(coords, field_values, title, config):
    """Scatter/contour plot of a single variable at a single timestep over the mesh."""
    fig, ax = plt.subplots(figsize=(10, 6))
    sc = ax.scatter(coords[:, 0], coords[:, 1], c=field_values, cmap="coolwarm",
                    s=5, alpha=0.8)
    plt.colorbar(sc, ax=ax, label=title)
    ax.set_title(title)
    ax.set_xlabel("x-coordinate")
    ax.set_ylabel("y-coordinate")
    ax.set_aspect("equal")
    plt.tight_layout()

    safe_title = title.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")
    out_path = os.path.join(config["output_dir"], f"spatial_{safe_title}.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved {out_path}")


def plot_prediction_vs_ground_truth(coords, predictions, ground_truth, scalers, timestep, config):
    """
    Side-by-side spatial plots for predicted vs actual at a given timestep.

    Parameters
    ----------
    predictions  : np.ndarray, shape (n_cells, n_steps, 3) — normalized
    ground_truth : np.ndarray, shape (n_cells, n_steps, 3) — normalized
    timestep     : int — which predicted step to visualize (0-indexed)
    """
    var_names = ["Pressure (P)", "U-velocity", "V-velocity"]
    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    fig.suptitle(f"Predicted vs Ground Truth at t+{timestep + 1}", fontsize=14)

    for vi, vname in enumerate(var_names):
        pred_denorm = scalers[vi].inverse_transform(
            predictions[:, timestep, vi].reshape(-1, 1)
        ).ravel()
        gt_denorm = scalers[vi].inverse_transform(
            ground_truth[:, timestep, vi].reshape(-1, 1)
        ).ravel()

        vmin = min(pred_denorm.min(), gt_denorm.min())
        vmax = max(pred_denorm.max(), gt_denorm.max())

        for ax, values, label in zip(axes[vi], [pred_denorm, gt_denorm],
                                      [f"Predicted {vname}", f"Ground Truth {vname}"]):
            sc = ax.scatter(coords[:, 0], coords[:, 1], c=values, cmap="coolwarm",
                            s=5, vmin=vmin, vmax=vmax, alpha=0.8)
            plt.colorbar(sc, ax=ax)
            ax.set_title(label)
            ax.set_xlabel("x")
            ax.set_ylabel("y")
            ax.set_aspect("equal")

    plt.tight_layout()
    out_path = os.path.join(config["output_dir"], f"pred_vs_gt_t{timestep + 1}.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved {out_path}")


def plot_error_histogram(predictions, ground_truth, scalers, config):
    """
    Histogram of prediction errors per variable.

    Parameters
    ----------
    predictions  : np.ndarray, shape (n_cells, n_steps, 3) — normalized
    ground_truth : np.ndarray, shape (n_cells, n_steps, 3) — normalized
    """
    var_names = ["Pressure (P)", "U-velocity", "V-velocity"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle("Prediction Error Distribution", fontsize=13)

    for vi, (ax, vname) in enumerate(zip(axes, var_names)):
        pred_denorm = scalers[vi].inverse_transform(
            predictions[:, :, vi].reshape(-1, 1)
        ).ravel()
        gt_denorm = scalers[vi].inverse_transform(
            ground_truth[:, :, vi].reshape(-1, 1)
        ).ravel()
        errors = pred_denorm - gt_denorm
        rmse = np.sqrt(np.mean(errors ** 2))
        ax.hist(errors, bins=50, edgecolor="black", alpha=0.75)
        ax.set_title(f"{vname}\nRMSE={rmse:.4f}")
        ax.set_xlabel("Error")
        ax.set_ylabel("Count")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    out_path = os.path.join(config["output_dir"], "error_histogram.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved {out_path}")


def plot_autoregressive_error_growth(predictions, ground_truth, scalers, config):
    """
    Line plot showing how RMSE grows with rollout step number.

    Parameters
    ----------
    predictions  : np.ndarray, shape (n_cells, n_steps, 3) — normalized
    ground_truth : np.ndarray, shape (n_cells, n_steps, 3) — normalized
    """
    n_steps = min(predictions.shape[1], ground_truth.shape[1])
    var_names = ["Pressure (P)", "U-velocity", "V-velocity"]

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.suptitle("Autoregressive RMSE Growth Over Rollout Steps", fontsize=13)

    for vi, vname in enumerate(var_names):
        rmse_per_step = []
        for t in range(n_steps):
            pred_denorm = scalers[vi].inverse_transform(
                predictions[:, t, vi].reshape(-1, 1)
            ).ravel()
            gt_denorm = scalers[vi].inverse_transform(
                ground_truth[:, t, vi].reshape(-1, 1)
            ).ravel()
            rmse = np.sqrt(np.mean((pred_denorm - gt_denorm) ** 2))
            rmse_per_step.append(rmse)
        ax.plot(range(1, n_steps + 1), rmse_per_step, marker="o", label=vname, linewidth=1.5)

    ax.set_xlabel("Rollout Step")
    ax.set_ylabel("RMSE (physical units)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    out_path = os.path.join(config["output_dir"], "autoregressive_rmse_growth.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved {out_path}")

## 9. Main Inference Pipeline

Orchestrates: data loading → model loading → prediction → coordinate queries → export → visualization

In [ ]:
def run_inference():
    """
    Orchestrate the full inference pipeline.

    Returns
    -------
    dict with keys:
      model               : nn.Module — the loaded model in eval mode
      single_step_predictions : np.ndarray (n_cells, pred_steps, 3) — normalized
      rollout_predictions : np.ndarray (n_cells, autoregressive_steps, 3) — normalized
      querier             : CoordinateQuery — for ad-hoc coordinate lookups
      scalers             : list of 3 StandardScaler — for denormalization
      coords              : np.ndarray (n_cells, 2) — mesh coordinates
      data_normalized     : np.ndarray (n_cells, n_timesteps, 3) — full normalized history
      neighbor_indices    : np.ndarray (n_cells, k) — kNN graph indices
    Returns None if the requested checkpoint is not found.
    """
    cfg = INFERENCE_CONFIG

    # ── 1. Load data ─────────────────────────────────────────────────────────────────
    print("=" * 60)
    print("STEP 1: Loading data")
    print("=" * 60)
    coords, data, n_timesteps, dataframes = parse_urans_multifile(
        cfg["pressure_csv"], cfg["u_velocity_csv"], cfg["v_velocity_csv"]
    )

    # ── 2. Build kNN graph ────────────────────────────────────────────────────────────
    print("\nSTEP 2: Building spatial graph")
    neighbor_indices, neighbor_distances = build_knn_graph(coords, cfg["k_neighbors"])

    # ── 3. Normalize (using training split statistics) ────────────────────────────
    print("\nSTEP 3: Normalizing data")
    train_end_t = int(cfg["train_ratio"] * n_timesteps)
    data_normalized, scalers = normalize_per_variable(data, train_end_t)

    # ── 4. Load model ─────────────────────────────────────────────────────────────────
    print(f"\nSTEP 4: Loading {cfg['active_model']} model")
    checkpoint_map = {
        "SpatioTemporalLSTM": cfg["lstm_checkpoint"],
        "SpatioTemporalTransformer": cfg["transformer_checkpoint"],
    }
    ckpt_path = checkpoint_map.get(cfg["active_model"])
    if ckpt_path is None or not os.path.exists(ckpt_path):
        print(f"  ⚠ Checkpoint not found: {ckpt_path}")
        print(f"  Available checkpoints:")
        for name, path in checkpoint_map.items():
            exists = path and os.path.exists(path)
            print(f"    {name}: {path} {'✓' if exists else '✗'}")
        return None

    model = load_trained_model(cfg["active_model"], ckpt_path, cfg, device)

    # ── 5. Single-step prediction ───────────────────────────────────────────────────────
    print("\nSTEP 5: Running single-step prediction (all cells)")
    single_preds = predict_all_cells(model, data_normalized, neighbor_indices, neighbor_distances, cfg, device)
    print(f"  Predictions shape: {single_preds.shape}")

    # ── 6. Autoregressive rollout ─────────────────────────────────────────────────────────
    print(f"\nSTEP 6: Autoregressive rollout ({cfg['autoregressive_steps']} steps)")
    rollout_preds = autoregressive_rollout(
        model, data_normalized, neighbor_indices, neighbor_distances, cfg, device,
        n_rollout_steps=cfg["autoregressive_steps"]
    )
    print(f"  Rollout predictions shape: {rollout_preds.shape}")

    # ── 7. Coordinate query system ───────────────────────────────────────────────────────
    print("\nSTEP 7: Setting up coordinate query system")
    querier = CoordinateQuery(coords, rollout_preds, scalers)

    # Auto-generate some interesting query points if none specified
    if not cfg["query_coordinates"]:
        # Sample 5 points from the mesh itself
        sample_indices = np.linspace(0, len(coords)-1, 5, dtype=int)
        cfg["query_coordinates"] = coords[sample_indices].tolist()
        print(f"  Auto-selected {len(cfg['query_coordinates'])} query points from mesh")

    # Run queries
    print("\n  Query results:")
    for qx, qy in cfg["query_coordinates"]:
        result = querier.query_nearest(qx, qy)
        p_pred = result["predictions"]["pressure"]
        print(f"    ({qx:.4f}, {qy:.4f}) → nearest cell {result['nearest_cell_idx']}, "
              f"dist={result['distance']:.6f}, P[t1]={p_pred[0]:.4f}")

    # ── 8. Export ────────────────────────────────────────────────────────────────────────
    if cfg["export_csv"]:
        print(f"\nSTEP 8: Exporting predictions to CSV")
        export_predictions_csv(coords, single_preds, scalers, cfg, prefix="single_step")
        export_predictions_csv(coords, rollout_preds, scalers, cfg, prefix="rollout")

    # ── 9. Visualizations ─────────────────────────────────────────────────────────────────
    if cfg["create_visualizations"]:
        print(f"\nSTEP 9: Creating visualizations")

        # 9a. Time-series at queried coordinates
        for i, (qx, qy) in enumerate(cfg["query_coordinates"][:3]):
            result = querier.query_nearest(qx, qy)
            plot_coordinate_prediction(result, cfg)

        # 9b. Spatial field at first predicted timestep
        for vi, vname in enumerate(["Pressure", "U-velocity", "V-velocity"]):
            # Denormalize first timestep predictions
            field = scalers[vi].inverse_transform(
                single_preds[:, 0, vi].reshape(-1, 1)
            ).ravel()
            plot_spatial_field(coords, field, f"Predicted {vname} (t+1)", cfg)

        # 9c. Autoregressive error growth (if ground truth available)
        val_end_t = int((cfg["train_ratio"] + cfg["val_ratio"]) * n_timesteps)
        test_start = val_end_t
        available_gt_steps = n_timesteps - test_start
        if available_gt_steps > 0:
            n_compare = min(cfg["autoregressive_steps"], available_gt_steps)
            # data is (n_cells, n_timesteps, 3), ground truth for test period
            # The single_step prediction uses the last seq_len timesteps, so ground truth starts at n_timesteps
            # But we don't have ground truth beyond n_timesteps!
            # For test evaluation, use the test split ground truth
            print("\n  Note: For error analysis, use the evaluate() function from training.")

        print(f"\n  All visualizations saved to {cfg['output_dir']}/")

    print("\n" + "=" * 60)
    print("INFERENCE COMPLETE")
    print("=" * 60)

    return {
        "model": model,
        "single_step_predictions": single_preds,
        "rollout_predictions": rollout_preds,
        "querier": querier,
        "scalers": scalers,
        "coords": coords,
        "data_normalized": data_normalized,
        "neighbor_indices": neighbor_indices,
        "neighbor_distances": neighbor_distances,
    }

In [ ]:
results = run_inference()

## 10. Interactive Coordinate Query

Query any (x, y) coordinate to get its predicted flow values.
Modify the coordinates below and re-run the cell.

In [ ]:
# ── MODIFY THESE COORDINATES ────────────────────────────────────────────────────────────────────────
query_points = [
    [0.5, 0.1],    # Example: near airfoil leading edge
    [1.0, 0.0],    # Example: trailing edge
    [0.3, 0.05],   # Example: upper surface
    [0.3, -0.05],  # Example: lower surface
    [2.0, 0.0],    # Example: wake region
]

if results is not None:
    querier = results["querier"]

    print("=" * 80)
    print(f"{'Coordinate':<20} {'Nearest Cell':<15} {'Distance':<12} {'P(t+1)':<12} {'U(t+1)':<12} {'V(t+1)':<12}")
    print("=" * 80)

    for qx, qy in query_points:
        r = querier.query_nearest(qx, qy)
        p = r["predictions"]["pressure"][0]
        u = r["predictions"]["u_velocity"][0]
        v = r["predictions"]["v_velocity"][0]
        print(f"({qx:>7.3f}, {qy:>7.3f})  "
              f"Cell {r['nearest_cell_idx']:<8d}  "
              f"{r['distance']:<11.6f}  "
              f"{p:<11.4f}  {u:<11.5f}  {v:<11.5f}")

    print("\n── Interpolated Query (k=5 nearest cells) ──")
    for qx, qy in query_points[:2]:
        r = querier.query_interpolated(qx, qy, k=5)
        p = r["predictions"]["pressure"][0]
        u = r["predictions"]["u_velocity"][0]
        v = r["predictions"]["v_velocity"][0]
        print(f"({qx:>7.3f}, {qy:>7.3f})  "
              f"P={p:<11.4f}  U={u:<11.5f}  V={v:<11.5f}  "
              f"(interpolated from cells {r['interpolation_cells']})")

## 11. Full Time-Series Prediction at a Coordinate

Plot the complete predicted P, U, V time series at a specific coordinate,
including both the known history and the predicted future.

In [ ]:
if results is not None:
    # Pick a coordinate to visualize
    qx, qy = query_points[0]

    # Find nearest cell
    tree = cKDTree(results["coords"])
    _, cell_idx = tree.query([qx, qy])

    # Get history (denormalized)
    history_norm = results["data_normalized"][cell_idx]  # (n_timesteps, 3)
    history_phys = np.zeros_like(history_norm)
    for vi in range(3):
        history_phys[:, vi] = results["scalers"][vi].inverse_transform(
            history_norm[:, vi].reshape(-1, 1)
        ).ravel()

    # Get predictions (denormalized)
    pred_norm = results["rollout_predictions"][cell_idx]  # (rollout_steps, 3)
    pred_phys = np.zeros_like(pred_norm)
    for vi in range(3):
        pred_phys[:, vi] = results["scalers"][vi].inverse_transform(
            pred_norm[:, vi].reshape(-1, 1)
        ).ravel()

    n_hist = history_phys.shape[0]
    n_pred = pred_phys.shape[0]

    var_names = ["Pressure (P)", "U-velocity", "V-velocity"]
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    fig.suptitle(f"Flow Prediction at ({qx:.4f}, {qy:.4f}) — Cell {cell_idx}", fontsize=14)

    for vi, (ax, vname) in enumerate(zip(axes, var_names)):
        t_hist = np.arange(n_hist)
        t_pred = np.arange(n_hist, n_hist + n_pred)

        ax.plot(t_hist, history_phys[:, vi], color="steelblue", linewidth=1.5, label="History")
        ax.plot(t_pred, pred_phys[:, vi], color="crimson", linewidth=2, linestyle="--", label="Prediction")
        ax.axvline(x=n_hist - 0.5, color="gray", linestyle=":", linewidth=1, label="Prediction start")

        ax.set_ylabel(vname)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("Timestep")
    plt.tight_layout()
    out_path = os.path.join(INFERENCE_CONFIG["output_dir"], f"full_timeseries_cell_{cell_idx}.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {out_path}")